# Suno API via `arioso`

## What this is

Suno has **no official public API**. Every programmatic access method works by
reverse-engineering Suno's private web endpoints and authenticating through
browser cookies extracted from a logged-in session. `arioso` wraps this
reverse-engineered flow into a clean Python interface.

## Fragilities you should know about

| Issue | Impact |
|-------|--------|
| **Cookies expire every ~7 days** | You'll need to re-extract your cookie regularly. |
| **hCaptcha on every generation** | Suno demands an hCaptcha token for every `generate`/`cover` call. Without it you get "Token validation failed". We solve this via the paid [2Captcha](https://2captcha.com) service (~$2.99/1,000 solves). |
| **Endpoint churn** | Suno has already migrated from `studio-api.suno.ai` to `studio-api.prod.suno.com`. Further changes can break things without warning. |
| **Clerk JS version drift** | The auth backend version string changes periodically, breaking token refresh. |
| **ToS violation** | Suno's Terms of Service explicitly prohibit automated access, scraping, and circumventing content protections. |
| **Free account limits** | Only 50 credits/day, 60-second max audio uploads, no commercial rights. |

## What you need

1. **A Suno cookie** (to authenticate as your Suno account)
2. **A 2Captcha API key** (to solve the mandatory hCaptcha on generation calls)

## How to get your cookie

### Option 1: Automatic extraction from Chrome (easiest)

If you're logged into [suno.com](https://suno.com) in Chrome, `arioso` can
grab the cookie automatically via `browser-cookie3`:

```python
client = SunoCookieClient(browser="chrome", twocaptcha_key="...")
```

**Caveat:** On macOS, the first time this runs, a Keychain dialog will ask
"python wants to access Chrome Safe Storage". Click **Always Allow**.
If the dialog appears in the background and goes unanswered, the call hangs.
Safari cookies are sandboxed and require Full Disk Access — Chrome is easier.

### Option 2: Manual extraction (more reliable)

1. Open Chrome and go to [suno.com/create](https://suno.com/create)
2. Open DevTools: **Cmd+Option+I** (Mac) or **F12** (Windows/Linux)
3. Go to the **Network** tab
4. Refresh the page (Cmd+R)
5. In the filter bar, type `clerk`
6. Click any request to `clerk.suno.com`
7. In the **Headers** pane, find the `Cookie:` request header
8. Copy the **entire value** (it's long — make sure you get all of it)

Then pass it directly:
```python
client = SunoCookieClient(cookie="__client=...; __session=...; ...", twocaptcha_key="...")
```

## How to get a 2Captcha key

1. Sign up at [2captcha.com](https://2captcha.com)
2. Add funds (minimum $3 — enough for ~1,000 hCaptcha solves)
3. Copy your API key from the dashboard
4. Pass it as `twocaptcha_key=` below, or set the `TWOCAPTCHA_KEY` env var

## Setup

Fill in your credentials below:
- **`cookie`**: Paste your cookie string, or leave as `None` to auto-extract from Chrome.
- **`twocaptcha_key`**: Your 2Captcha API key (required for generation/cover calls).

In [ ]:
# ── Credentials ─────────────────────────────────────────────────────
# Paste your cookie string here, or leave as None to auto-extract from Chrome.
cookie = None  # e.g. "__client=abc123; __session=xyz789; ..."

# Your 2Captcha API key — required for generate/cover calls.
# Get one at https://2captcha.com (~$2.99 for 1,000 hCaptcha solves).
twocaptcha_key = None  # e.g. "abc123def456..."

# ── Build the client ────────────────────────────────────────────────
from arioso.suno import SunoCookieClient

if cookie:
    client = SunoCookieClient(cookie=cookie, twocaptcha_key=twocaptcha_key)
else:
    # Tries to grab cookies from Chrome automatically.
    # On first run, macOS will prompt: "python wants to access Chrome Safe Storage"
    # → click "Always Allow".
    client = SunoCookieClient(browser="chrome", twocaptcha_key=twocaptcha_key)

print("Client created. Running diagnostics...")
report = client.diagnose()
for k, v in report.items():
    print(f"  {k}: {v}")

if not twocaptcha_key:
    print("\n⚠ No 2Captcha key provided.")
    print("  Upload/feed will work, but generate/cover will fail with")
    print("  'Token validation failed' (Suno requires hCaptcha on every generation).")

## Generate a jazz cover from a local audio file

This demo uploads a local WAV file to Suno, then generates a jazz
accompaniment from it using the "cover" feature.

The flow is:
1. **Upload** the audio file to Suno's servers (goes through S3).
2. **Generate** a cover using the uploaded clip as the source, with style/weight controls.
3. **Poll** until the generation is complete.
4. **Download** the result.

In [2]:
# ── Step 1: Upload the source audio ─────────────────────────────────

audio_file = "/Users/thorwhalen/Downloads/6dim.wav"  # path to your audio file

print(f"Uploading {audio_file}...")
clip_id = client.upload_audio(audio_file)
print(f"Upload complete. Clip ID: {clip_id}")

Uploading /Users/thorwhalen/Downloads/6dim.wav...


HTTPError: 422 Client Error: Unprocessable Entity for url: https://studio-api.prod.suno.com/api/uploads/audio/74b4d7d4-a6ae-4db2-af44-fc6f722eab41/upload-finish/

In [ ]:
# ── Step 2: Generate a jazz cover ───────────────────────────────────

songs = client.cover(
    clip_id,                        # UUID of the uploaded audio
    style=(                         # genre/style tags guiding the generation
        "Bass, Piano and Drums Jazz accompaniment, at 100 bpms"
    ),
    title="6dim Jazz Cover",        # title for the generated track
    instrumental=True,              # no vocals — pure accompaniment
    style_weight=0.80,              # 0-1: how strictly to follow the style tags
    # audio_weight is controlled by Suno internally via the cover task;
    # weirdness_constraint controls creative variation:
    weirdness_constraint=0.20,      # 0-1: low = conservative, high = experimental
    negative_tags="Rock, EDM, Electronic, Heavy Metal",  # styles to avoid
    model="chirp-crow",             # v5 model (latest as of March 2026)
)

print(f"Generation started — {len(songs)} song(s):")
for s in songs:
    print(f"  ID: {s.id}  |  Title: {s.title!r}  |  Status: {s.status}")

In [ ]:
# ── Step 3: Wait for completion ─────────────────────────────────────

print("Waiting for generation to complete (this may take 1-3 minutes)...")
songs = client.wait_for_songs(
    songs,
    timeout=300,        # max 5 minutes
    poll_interval=5,    # check every 5 seconds
)

for s in songs:
    print(f"  {s.title!r}  |  Status: {s.status}  |  URL: {s.audio_url[:60]}...")

In [ ]:
# ── Step 4: Download and play ───────────────────────────────────────

song = client.download_song(songs[0])
print(f"Downloaded {len(song.audio_bytes):,} bytes")

# Save to disk
output_path = "/Users/thorwhalen/Downloads/6dim_jazz_cover.mp3"
with open(output_path, "wb") as f:
    f.write(song.audio_bytes)
print(f"Saved to {output_path}")

# Play in notebook (if IPython/Jupyter is available)
try:
    from IPython.display import Audio, display
    display(Audio(data=song.audio_bytes, rate=None, autoplay=False))
except ImportError:
    print("(IPython not available — open the saved file to listen)")